# Smart Agriculture – vollständiges Projektnotebook

Dieses Notebook bearbeitet die gesamte Aufgabe: Projektplanung, Datengenerierung, API-Design, Datenanalyse, Cybersecurity, App-Konzept und Abschlussdokumentation. Die `README.md` bleibt unverändert.

## Übersicht der Phasen

1. Projektplanung
2. Datengenerierung & CSV-Export
3. API-Design & Minimaler FastAPI-Prototyp
4. Datenanalyse und Prozessfähigkeit
5. Cybersecurity Review
6. Flutter-App-Konzept
7. Abschlussdokumentation und Präsentation

In [ ]:
import importlib.util
import subprocess
import sys

packages = ['numpy', 'pandas', 'matplotlib', 'fastapi', 'uvicorn', 'python-multipart', 'python-pptx']
for package in packages:
    if importlib.util.find_spec(package) is None:
        print(f'Installiere fehlendes Paket: {package}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
    else:
        print(f'Paket vorhanden: {package}')

## Phase 1 – Projektplanung

**Projektname:** Smart Agriculture Monitoring

**Zielsetzung:** Entwicklung eines digitalen Überwachungssystems für ein Gewächshaus, das Sensordaten simuliert, analysiert, über eine API bereitstellt und in einer Windows-Desktop-App visualisiert.

**Stakeholder:** Betriebsleitung, Agraringenieure, IT-Verantwortliche, Lehrperson, Endanwender.

**Systemgrenzen:** Enthalten sind Simulation, CSV-Export, API, Datenanalyse und Desktop-App. Ausgeschlossen sind echte Sensoren, physische Steuerung und Cloud-Deployment.

**Funktionale Anforderungen:**
- Mindestens 100 simulierte Messwerte
- CSV-Ausgabe der Messwerte
- API-Endpunkte für Sensorwerte, Statistik, Alerts und Status
- Windows-Dashboard mit aktuellen Werten, Kennzahlen und Warnungen

**Nichtfunktionale Anforderungen:**
- Antwortzeit der API < 500 ms
- Reproduzierbarkeit der Datenerzeugung
- Klare Dokumentation
- Eingabevalidierung und Sicherheitsüberlegungen

## Phase 2 – Datengenerierung & Datenmodell

Datenmodell:
- `timestamp`
- `soil_moisture_percent`
- `soil_temperature_c`
- `air_temperature_c`
- `air_humidity_percent`
- `ph_value`
- `irrigation_active`
- `system_status`

Qualitätsgrenzen für pH:
- LSL = 5.8
- Zielwert = 6.5
- USL = 7.2

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

def create_simulated_data(output_path='smart_agriculture_measurements.csv', n=120):
    np.random.seed(42)
    start = datetime.now().replace(second=0, microsecond=0)
    timestamps = [start + timedelta(minutes=5 * i) for i in range(n)]

    soil_moisture = np.random.normal(loc=42, scale=6, size=n)
    soil_temperature = np.random.normal(loc=18, scale=2.5, size=n)
    air_temperature = np.random.normal(loc=23, scale=3, size=n)
    air_humidity = np.random.normal(loc=58, scale=8, size=n)
    ph_value = np.random.normal(loc=6.5, scale=0.22, size=n)

    ph_value[40] = 5.4
    ph_value[85] = 7.5

    irrigation_active = soil_moisture < 36
    system_status = np.where((ph_value < 5.8) | (ph_value > 7.2), 'ALERT', 'OK')

    df = pd.DataFrame({
        'timestamp': timestamps,
        'soil_moisture_percent': np.round(soil_moisture, 2),
        'soil_temperature_c': np.round(soil_temperature, 2),
        'air_temperature_c': np.round(air_temperature, 2),
        'air_humidity_percent': np.round(air_humidity, 2),
        'ph_value': np.round(ph_value, 2),
        'irrigation_active': irrigation_active,
        'system_status': system_status,
    })
    df.to_csv(output_path, index=False)
    return df

df = create_simulated_data()
print('CSV-Datei erstellt: smart_agriculture_measurements.csv')
df.head()

In [ ]:
df.describe(include='all')

## Phase 3 – API-Design

### Endpunkte
- `GET /health` – Statusprüfung
- `GET /sensors` – Aktuelle Sensordaten
- `GET /statistics` – Statistische Kennzahlen
- `GET /alerts` – Warnmeldungen
- `POST /login` – Authentifizierung

### Beispielantworten
**GET /health**
```json
{
  "status": "ok"
}
```

**GET /statistics**
```json
{
  "ph_mean": 6.51,
  "ph_std": 0.22,
  "cp": 1.09,
  "cpk": 0.98
}
```

In [ ]:
from pathlib import Path

api_code = '''from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel
import pandas as pd

DATA_PATH = 'smart_agriculture_measurements.csv'
oath2_scheme = OAuth2PasswordBearer(tokenUrl='/login')
app = FastAPI(title='Smart Agriculture API')

class HealthResponse(BaseModel):
    status: str

class SensorData(BaseModel):
    timestamp: str
    soil_moisture_percent: float
    soil_temperature_c: float
    air_temperature_c: float
    air_humidity_percent: float
    ph_value: float
    irrigation_active: bool
    system_status: str

class StatisticsResponse(BaseModel):
    ph_mean: float
    ph_median: float
    ph_variance: float
    ph_std: float
    cp: float
    cpk: float

def load_data():
    return pd.read_csv(DATA_PATH, parse_dates=['timestamp'])

def calculate_statistics(df):
    ph = df['ph_value']
    lsl = 5.8
    usl = 7.2
    mean_ph = float(ph.mean())
    median_ph = float(ph.median())
    var_ph = float(ph.var(ddof=1))
    std_ph = float(ph.std(ddof=1))
    cp = float((usl - lsl) / (6 * std_ph)) if std_ph > 0 else 0.0
    cpk = float(min((usl - mean_ph) / (3 * std_ph), (mean_ph - lsl) / (3 * std_ph))) if std_ph > 0 else 0.0
    return {
        'ph_mean': round(mean_ph, 3),
        'ph_median': round(median_ph, 3),
        'ph_variance': round(var_ph, 4),
        'ph_std': round(std_ph, 4),
        'cp': round(cp, 3),
        'cpk': round(cpk, 3),
    }

@app.get('/health', response_model=HealthResponse)
def health():
    return {'status': 'ok'}

@app.post('/login')
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    if form_data.username != 'admin' or form_data.password != 'password':
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail='Ungültige Anmeldedaten', headers={'WWW-Authenticate': 'Bearer'})
    return {'access_token': 'demo-token', 'token_type': 'bearer'}

@app.get('/sensors')
def sensors(token: str = Depends(lambda token: token)):
    df = load_data()
    return df.to_dict(orient='records')

@app.get('/statistics', response_model=StatisticsResponse)
def statistics(token: str = Depends(lambda token: token)):
    df = load_data()
    return calculate_statistics(df)

@app.get('/alerts')
def alerts(token: str = Depends(lambda token: token)):
    df = load_data()
    alerts = []
    for _, row in df.iterrows():
        if row['system_status'] == 'ALERT':
            alerts.append({'timestamp': row['timestamp'], 'type': 'PH_OUT_OF_SPEC', 'severity': 'HIGH', 'message': 'pH-Wert außerhalb der Spezifikation'})
    return alerts
'''

Path = Path('smart_agriculture_api.py')
Path.write_text(api_code)
print('Die Datei smart_agriculture_api.py wurde erstellt.')

## Phase 4 – Datenanalyse

Berechne Mittelwert, Median, Varianz, Standardabweichung sowie Cp/Cpk für den pH-Wert.

In [ ]:
ph = df['ph_value']
lsl = 5.8
usl = 7.2
mean_ph = ph.mean()
median_ph = ph.median()
var_ph = ph.var(ddof=1)
std_ph = ph.std(ddof=1)
cp = (usl - lsl) / (6 * std_ph)
cpk = min((usl - mean_ph) / (3 * std_ph), (mean_ph - lsl) / (3 * std_ph))
result = {
    'mean_ph': round(mean_ph, 3),
    'median_ph': round(median_ph, 3),
    'variance_ph': round(var_ph, 4),
    'std_ph': round(std_ph, 4),
    'cp': round(cp, 3),
    'cpk': round(cpk, 3),
}
result

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(ph, bins=15, color='skyblue', edgecolor='black')
plt.axvline(lsl, color='red', linestyle='--', label='LSL')
plt.axvline(usl, color='red', linestyle='--', label='USL')
plt.axvline(mean_ph, color='green', linestyle='-', label='Mittelwert')
plt.title('Histogramm des pH-Werts')
plt.xlabel('pH-Wert')
plt.ylabel('Häufigkeit')
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df['timestamp'], ph, marker='o', linestyle='-', markersize=4)
plt.axhline(lsl, color='red', linestyle='--', label='LSL')
plt.axhline(usl, color='red', linestyle='--', label='USL')
plt.axhline(mean_ph, color='green', linestyle='-', label='Mittelwert')
plt.title('Zeitverlauf des pH-Werts')
plt.xlabel('Zeit')
plt.ylabel('pH-Wert')
plt.xticks(rotation=45)
plt.tight_layout()
plt.legend()
plt.show()

## Phase 5 – Cybersecurity Review

Fülle die Risikomatrix aus und dokumentiere Mindestmaßnahmen.

| Nr. | Schwachstelle | Risiko | Eintritt | Auswirkung | Maßnahme |
|----|---------------|--------|----------|------------|----------|
| 1 | Keine Authentifizierung | Unbefugter Zugriff | hoch | hoch | Login und Token nutzen |
| 2 | Fehlende Validierung | Manipulierte Anfragen | mittel | hoch | Pydantic-Modelle nutzen |
| 3 | Hardcodierte Secrets | Geheimnisverlust | mittel | hoch | Secrets auslagern |
| 4 | Keine HTTPS/Verschlüsselung | Abgriff von Daten | mittel | mittel | TLS einsetzen |
| 5 | Kein Logging | Vorfälle schwer nachverfolgen | niedrig | mittel | Logging implementieren |

## Phase 6 – Flutter-App-Konzept

**Mindestfunktionen:**
- Darstellung aktueller Sensordaten
- Anzeige von Mittelwert, Standardabweichung, Cp und Cpk
- Warnungen und Systemstatus

**Empfohlene Screens:**
1. Dashboard
2. Analyse
3. Alerts / Security
4. Projektinfo

**Beispielhafte Widget-Struktur:**
```dart
Column(
  children: [
    Text('Smart Agriculture Dashboard'),
    Text('pH Mittelwert: 6.51'),
    Text('Cpk: 0.98'),
    Text('Status: Warning'),
  ],
)
```

## Phase 7 – Abschlussdokumentation

Beschreibe die wichtigsten Ergebnisse, das Datenmodell, das API-Design, die Analyse, das Cybersecurity-Review und die App-Planung.

**Offene Punkte:**
- Integration echter Sensoren
- Produktionstaugliche Authentifizierung
- Cloud-Deployment
- Historische Trendanalysen

In [ ]:
from pptx import Presentation
from pptx.util import Pt

slides = [
    {'title': 'Smart Agriculture Projekt', 'content': ['Digitales Überwachungssystem für ein Gewächshaus', 'Simulation, Analyse, API und Windows-Desktop-App']},
    {'title': 'Projektziel', 'content': ['Sensordaten simulieren und analysieren', 'REST-API entwerfen und dokumentieren', 'Ergebnisse in einer Windows-App visualisieren']},
    {'title': 'Systemarchitektur', 'content': ['Datenmodell: CSV-Daten für Sensorwerte', 'API: FastAPI mit /health, /sensors, /statistics, /alerts, /login', 'App: Windows-Desktop-Dashboard']},
    {'title': 'Aktueller Umsetzungsstand', 'content': ['Datensimulation mit 120 Messwerten erzeugt', 'CSV-Datei bereit zur Analyse', 'FastAPI-Code und Projektplan erstellt']},
    {'title': 'Analyseergebnisse', 'content': ['pH-Statistik: Mittelwert, Median, Varianz, Std Dev', 'Prozessfähigkeit: Cp und Cpk berechnet', 'Alarmstatus bei pH außerhalb von 5.8 - 7.2']},
    {'title': 'Cybersecurity & nächste Schritte', 'content': ['Risiken: Authentifizierung, Validierung, HTTPS, Logging', 'Gegenmaßnahmen geplant: Token, Pydantic, sichere Konfiguration', 'Nächster Schritt: echte Sensoranbindung und Cloud-Deployment']}
]

prs = Presentation()
for slide_info in slides:
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = slide_info['title']
    text_frame = slide.shapes.placeholders[1].text_frame
    text_frame.text = slide_info['content'][0]
    for line in slide_info['content'][1:]:
        paragraph = text_frame.add_paragraph()
        paragraph.text = line
        paragraph.level = 0
        paragraph.font.size = Pt(18)
output_file = 'Smart_Agriculture_Presentation.pptx'
prs.save(output_file)
print(f'Präsentation erstellt: {output_file}')